# preborn_config — DatasetConfigs for the PREBORN synthetic pipeline

Two independent MOSTLY AI generators, each a self-contained `DatasetConfig`
(group="core" only — PREBORN has NO reference tables):

* **PREBORN_MOTHER_CONFIG** — subject=person(mother): person → pregnancy,
  visit_occurrence, measurement, observation, condition_occurrence,
  procedure_occurrence. All facts are **person-rooted** (context_fk=person_id).
* **PREBORN_INFANT_CONFIG** — subject=person(infant): person → infant_attr
  (1:1 birth attributes, base_rate table), visit_occurrence, measurement,
  observation, condition_occurrence, procedure_occurrence.

`pregnancy_id` / `visit_occurrence_id` / `site_id` are dropped from training
and re-derived deterministically in `preborn_derive` (Phase B) on the synthetic
base, reusing PREBORN's own CDM logic (`sp1_msds_omop/.../preborn/_common.py`).

Real data source = the published `5_projects.preborn` (bare CDM tables).
Working (per-generator) schemas = `8_dev.synth_preborn_m` / `_i`.
Final synthetic tables = `5_projects.preborn.synth_*`.

In [0]:
%run /Workspace/Shared/ADC-DB/Prod/Synthetic/engine

In [0]:
# ============================================================================
# Locations / constants
# ============================================================================
SRC_CATALOG = "5_projects"
SRC_SCHEMA  = "preborn"                             # published real CDM tables
SRC = f"{SRC_CATALOG}.{SRC_SCHEMA}"

WORKING_CATALOG = "8_dev"
MOTHER_SCHEMA   = "synth_preborn_m"                 # per-generator working schema
INFANT_SCHEMA   = "synth_preborn_i"

# Final destination for the assembled synthetic copy: synth_* beside the real tables.
FINAL_CATALOG = "5_projects"
FINAL_SCHEMA  = "preborn"
SYNTH_PREFIX  = "synth_"

SITE_ID = "R1H"                                     # PREBORN is single-site (Barts)
RELEASE_DATE = "2026-07-15"                          # synthetic release stamp

# Real per-role sizes (targets; DP-modelled → validated within tolerance, not exact)
TARGET = {
    "mothers": 106_988, "infants": 98_135, "pregnancies": 136_996,
    "person": 205_123, "observation_period": 205_123,
    "visit_occurrence": 1_043_203, "measurement": 980_095, "observation": 2_582_375,
    "condition_occurrence": 509_036, "procedure_occurrence": 203_757,
    "episode": 136_973, "episode_event": 5_175_594, "fact_relationship": 237_084,
    "visit_detail": 9_045,
}

# Real per-role fact counts — the calibration targets for preborn_derive's
# downsample-to-real step (downsample-only; never fabricates rows).
TARGET_BY_ROLE = {
    "visit_occurrence":     {"mother": 1_034_158, "infant": 9_045},
    "measurement":          {"mother":   750_212, "infant": 229_883},
    "observation":          {"mother": 2_360_351, "infant": 222_024},
    "condition_occurrence": {"mother":   497_544, "infant": 11_492},
    "procedure_occurrence": {"mother":   203_589, "infant": 168},
}

# CDM field concepts used by episode_event (from _common.py:1074) and
# fact_relationship (from _common.py:1026). Kept here so preborn_derive reuses them.
FC_COND, FC_MEAS, FC_OBS, FC_PROC, FC_VIS = 1147127, 1147138, 1147165, 1147082, 1147070
EPISODE_EVENT_FIELDS = {                             # source_table -> (field concept, id col)
    "condition_occurrence": (FC_COND, "condition_occurrence_id"),
    "measurement":          (FC_MEAS, "measurement_id"),
    "observation":          (FC_OBS,  "observation_id"),
    "procedure_occurrence": (FC_PROC, "procedure_occurrence_id"),
    "visit_occurrence":     (FC_VIS,  "visit_occurrence_id"),
}
PERSON_DOMAIN = 1147026                              # person.person_id CDM field concept
REL_MOTHER, REL_CHILD, REL_TWIN, REL_SIBLING = 4248584, 4285883, 4013484, 4292398
EPISODE_CONCEPT = 32533                             # Disease Episode
EPISODE_OBJECT_PREGNANCY = 4299535                  # Pregnancy
EPISODE_TYPE = 32817                                # EHR
OBS_PERIOD_TYPE = 32817
VISIT_DETAIL_TYPE = 32817
VISIT_INPATIENT = 9201                              # infant nnu parent visit
VISIT_DETAIL_CONCEPT = 0                            # proprietary
CDM_VERSION_CONCEPT = 756265                        # OMOP CDM v5.4.0

# ============================================================================
# Column classification (read the real schema once; rule-based train/drop split)
# ============================================================================
def _cols(tbl):
    return [f.name for f in spark.table(f"{SRC}.{tbl}").schema.fields]

COLS = {t: _cols(t) for t in
        ["person", "pregnancy", "infant", "visit_occurrence", "measurement",
         "observation", "condition_occurrence", "procedure_occurrence"]}

# Columns never trained (assigned in Phase B, constant, or surrogate ids).
_LINEAGE_DROP = {
    "site_id", "pregnancy_id", "visit_occurrence_id", "visit_detail_id",
    "provider_id", "care_site_id", "location_id", "source_table",
    "enrichment_tag", "measurement_time", "person_role", "visit_source_value",
}

def _drops(src_tbl, extra=()):
    """drop_cols for a spec: lineage/admin cols + event-link cols + explicit extras
    that actually exist on the source table. pk/context/self are handled by the engine."""
    cs = COLS[src_tbl]
    d = [c for c in cs
         if c in _LINEAGE_DROP or c.endswith("_event_id") or c.endswith("_event_field_concept_id")]
    d += [c for c in extra if c in cs]
    return sorted(set(d))

# Date columns per table (drive temporal repair in assemble).
_DATES = {
    "person": ["birth_datetime"],
    "pregnancy": ["pregnancy_start_date", "pregnancy_end_date"],
    "visit_occurrence": ["visit_start_date", "visit_start_datetime",
                         "visit_end_date", "visit_end_datetime"],
    "measurement": ["measurement_date", "measurement_datetime"],
    "observation": ["observation_date", "observation_datetime"],
    "condition_occurrence": ["condition_start_date", "condition_start_datetime",
                             "condition_end_date", "condition_end_datetime"],
    "procedure_occurrence": ["procedure_date", "procedure_datetime",
                             "procedure_end_date", "procedure_end_datetime"],
}

# PII: nulled at write + hard-gated 100% NULL. person_source_value encodes
# site|idkind|patient-id -> identifying. Infant free-text handled via drop_cols.
_PII = {"person": ["person_source_value"]}

# ============================================================================
# Shared spec builders
# ============================================================================
def _person_spec():
    return TableSpec(name="person", group="core", pk="person_id",
                     date_cols=_DATES["person"], drop_cols=_drops("person"))

def _fact_spec(name):
    """A person-rooted clinical fact table."""
    pk = "visit_occurrence_id" if name == "visit_occurrence" else f"{name}_id"
    self_fk = "preceding_visit_occurrence_id" if name == "visit_occurrence" else None
    return TableSpec(name=name, group="core", pk=pk,
                     context_fk=("person_id", "person"), self_fk=self_fk,
                     date_cols=_DATES[name], drop_cols=_drops(name))

_FACT_TABLES = ["visit_occurrence", "measurement", "observation",
                "condition_occurrence", "procedure_occurrence"]

# ============================================================================
# Shared config kwargs
# ============================================================================
_DP = dict(enabled=True, max_epsilon=2.0, value_protection_epsilon=3.0, delta=1e-6,
           noise_multiplier=1.5, max_grad_norm=1.0)
_PROFILER = dict(epsilon_profiler=1.0, seqlen_parent_cap=5, cat_per_patient_cap=100,
                 seqlen_bins=[0, 1, 2, 3, 5, 10, 25, 50, 100, 1_000_000])
_RARE = dict(seqlen_support_min=0.05, rate_tol=0.02, privacy_safe_support=20, min_expected=5)

def _base_extra(role, output_persons):
    return dict(
        role=role,
        src_persons=output_persons,
        cohort=dict(persons=output_persons, seed=42, per_visit_keep=100,
                    meas_cap=5000, obs_cap=5000),
        output=dict(persons=output_persons),
        max_pandas_rows=15_000_000, max_pandas_gb=12.0,
        spark_date_min="1900-01-01", spark_date_max="2099-12-31",
        sdtypes_table="6_mgmt.default.sdtypes", sdtypes_prefix="preborn_",
        site_id_constant=SITE_ID,
        final=dict(catalog=FINAL_CATALOG, schema=FINAL_SCHEMA, prefix=SYNTH_PREFIX),
        validate=dict(
            fidelity_ratios=dict(
                visits_per_person="visit_occurrence",
                measurements_per_person="measurement",
                observations_per_person="observation",
                conditions_per_person="condition_occurrence",
                procedures_per_person="procedure_occurrence"),
        ),
    )

# ============================================================================
# MOTHER config
# ============================================================================
_MOTHER_TABLES = {"person": _person_spec()}
_MOTHER_TABLES["pregnancy"] = TableSpec(
    name="pregnancy", group="core", pk="pregnancy_id",
    context_fk=("person_id", "person"), date_cols=_DATES["pregnancy"],
    drop_cols=_drops("pregnancy"))
for _t in _FACT_TABLES:
    _MOTHER_TABLES[_t] = _fact_spec(_t)

_mother_extra = _base_extra("mother", TARGET["mothers"])
_mother_extra["validate"]["fidelity_ratios"]["pregnancies_per_person"] = "pregnancy"

PREBORN_MOTHER_CONFIG = DatasetConfig(
    name="preborn_mother",
    source={"catalog": SRC_CATALOG, "schema": SRC_SCHEMA},
    target={"catalog": WORKING_CATALOG, "schema": MOTHER_SCHEMA,
            "volume": f"/Volumes/{WORKING_CATALOG}/{MOTHER_SCHEMA}/artifacts"},
    subject="person", privacy_unit="person",
    tables=_MOTHER_TABLES,
    table_order=["person", "pregnancy"] + _FACT_TABLES,
    dp=_DP, profiler=_PROFILER, rare=_RARE,
    reference_chain=None,
    lifespan_rule={"birth_table": "person", "birth_col": "birth_datetime",
                   "death_table": None, "death_col": None},
    pii=_PII,
    pk_band_size=1_000_000_000,
    model_limits={"max_training_time": 120, "max_epochs": 50},
    enable_model_report=False,
    extra=_mother_extra,
)

# ============================================================================
# INFANT config
# ============================================================================
_INFANT_TABLES = {"person": _person_spec()}
# infant_attr: 1:1 birth attributes per infant person. base_rate_table => dedups to
# <=1 row per subject and needs no pk (death-style pure-linked). source_table='infant'
# for schema/PII lookup; the cohort builder supplies person_id (= infant_id).
_INFANT_TABLES["infant_attr"] = TableSpec(
    name="infant_attr", group="core", pk=None, base_rate_table=True,
    context_fk=("person_id", "person"), source_table="infant", date_cols=[],
    drop_cols=_drops("infant", extra=["infant_id", "labour_delivery_id",
                                      "birth_weight_source", "birth_apgar_source"]))
for _t in _FACT_TABLES:
    _INFANT_TABLES[_t] = _fact_spec(_t)

PREBORN_INFANT_CONFIG = DatasetConfig(
    name="preborn_infant",
    source={"catalog": SRC_CATALOG, "schema": SRC_SCHEMA},
    target={"catalog": WORKING_CATALOG, "schema": INFANT_SCHEMA,
            "volume": f"/Volumes/{WORKING_CATALOG}/{INFANT_SCHEMA}/artifacts"},
    subject="person", privacy_unit="person",
    tables=_INFANT_TABLES,
    table_order=["person", "infant_attr"] + _FACT_TABLES,
    dp=_DP, profiler=_PROFILER, rare=_RARE,
    reference_chain=None,
    lifespan_rule={"birth_table": "person", "birth_col": "birth_datetime",
                   "death_table": None, "death_col": None},
    pii=_PII,
    pk_band_size=1_000_000_000,
    model_limits={"max_training_time": 120, "max_epochs": 50},
    enable_model_report=False,
    extra=_base_extra("infant", TARGET["infants"]),
)

# ============================================================================
# Validate both configs
# ============================================================================
_em = validate_config(PREBORN_MOTHER_CONFIG)
_ei = validate_config(PREBORN_INFANT_CONFIG)
print("PREBORN_MOTHER_CONFIG:", len(PREBORN_MOTHER_CONFIG.tables), "tables | errors:", _em)
print("PREBORN_INFANT_CONFIG:", len(PREBORN_INFANT_CONFIG.tables), "tables | errors:", _ei)
print("mother table_order:", topo_order(PREBORN_MOTHER_CONFIG))
print("infant table_order:", topo_order(PREBORN_INFANT_CONFIG))
assert not _em and not _ei, "config validation failed — see errors above"